In [ ]:
import matplotlib.pyplot as plt
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()


# trajectory of one flight (prc770822360)
q("""
SELECT
  TIMESTAMP_MICROS(DIV(`timestamp`, 1000)) AS ts,
  altitude,
  groundspeed,
  vertical_rate,
  mach,
  source
FROM `prc-data-challenge-2025.prc2025.flights_train`
WHERE flight_id = 'prc770822360'
ORDER BY `timestamp`
""")

In [ ]:
# trajectory of one flight + altitude plot
df = q("""
SELECT
  TIMESTAMP_MICROS(DIV(`timestamp`, 1000)) AS ts,
  altitude,
  groundspeed,
  vertical_rate,
  mach,
  source
FROM `prc-data-challenge-2025.prc2025.flights_train`
WHERE flight_id = 'prc770822360'
ORDER BY `timestamp`
""")

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df["ts"], df["altitude"], ".", label="Altitude", markersize=1)
ax.set_xlabel("Time")
ax.set_ylabel("Altitude (ft)")
ax.legend()
plt.show()

We can se prc770822360 (12 h 18): trajectory has gaps of ~3 h 40, ~3 h 25 and ~1 h -> only ~half of the flight is covered.
Cause: ADS-B needs ground antennas (no coverage over oceans / some regions).
Starts 1 min after takeoff, ends on the runway. altitude is barometric (375 ft on ground at EHAM, elevation -11 ft).
-> key question: are gaps frequent across all flights?

In [ ]:
# largest gaps of one flight (prc770822360)
q(
    """
    SELECT TIMESTAMP_MICROS(DIV(`timestamp`, 1000)) AS ts,
     (`timestamp` - LAG(`timestamp`) OVER (PARTITION BY flight_id ORDER BY `timestamp`))
      / 1e9 AS gap_s,
    FROM `prc-data-challenge-2025.prc2025.flights_train`
    WHERE flight_id = 'prc770822360'
    ORDER BY gap_s DESC 
    LIMIT 20
"""
)

The list of the 20 longuest gaps in s.

In [ ]:
# gaps per flight (all flights)
df_gaps = q("""
WITH gaps AS (
  SELECT
    flight_id,
    `timestamp`,
    (`timestamp` - LAG(`timestamp`) OVER (PARTITION BY flight_id ORDER BY `timestamp`)) 
    / 1e9 AS gap_s
  FROM `prc-data-challenge-2025.prc2025.flights_train`
)
SELECT
  flight_id,
  COUNT(*) AS n_points,
  (MAX(`timestamp`) - MIN(`timestamp`)) / 1e9 AS span_s,
  MAX(gap_s) AS max_gap_s,
  SUM(IF(gap_s > 600, gap_s, 0)) AS total_gap_s
FROM gaps
GROUP BY flight_id
""")
df_gaps["gap_ratio"] = df_gaps["total_gap_s"] / df_gaps["span_s"]

In [ ]:
# check gaps on prc770822360
df_gaps[df_gaps["flight_id"] == "prc770822360"]

df_gaps: one row per flight, computed from flights_train (LAG on timestamp, per flight).
- n_points: number of trajectory measurements
- span_s: time between first and last measurement (s)
- max_gap_s: largest time between two consecutive measurements (s)
- total_gap_s: sum of gaps > 600 s, i.e. time with no data (s)
- gap_ratio: total_gap_s / span_s, share of the trajectory with no data
Check on prc770822360: span 12 h 17, max gap 3 h 43, total gaps 8 h 10, gap_ratio 0.66 -> matches the plot.

In [ ]:
# gap stats over all flights
df_gaps.describe()

gap_ratio over all 11037 flights: Q1 = 0, median 0.24, Q3 0.57, max 0.97 (mean 0.31).
-> >= 25% of flights fully covered, but half lose >= 24% of their trajectory, a quarter lose > 57%.
prc770822360 (0.66) is among the worst ~20%, not an exception.
~124M trajectory points in total (mean 11200 per flight).
Some trajectories are tiny (min span 170 s, 17 points).
-> gaps are a central issue: need a strategy for fuel intervals with no trajectory data.

In [ ]:
# gap_ratio vs trajectory span (plot)
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    df_gaps["span_s"] / 3600, df_gaps["gap_ratio"], ".", label="gap_ratio", markersize=1
)
ax.set_xlabel("span_s (h)")
ax.set_ylabel("gap_ratio")
ax.legend()
plt.show()

gap_ratio vs trajectory span (h): longer flights have more gaps.
Fully covered flights (gap_ratio = 0) only up to ~6.5 h; above 7 h most flights lose 40-90% of their trajectory.
Lower-left curve = artefact of the 600 s threshold (min non-zero ratio = 600 / span_s).
Dense clusters (e.g. ~4 h / 0.3, ~9 h / 0.8): to explain.

In [ ]:
# null ratio per column
q(
    """
    SELECT COUNTIF(altitude IS NULL) / COUNT(*) AS null_altitude_ratio,
    COUNTIF(groundspeed IS NULL) / COUNT(*) AS null_groundspeed_ratio,
    COUNTIF(vertical_rate IS NULL) / COUNT(*) AS null_vertical_rate_ratio,
    COUNTIF(mach IS NULL) / COUNT(*) AS null_mach_ratio,
    COUNTIF(track IS NULL) / COUNT(*) AS null_track_ratio,
    COUNTIF(CAS IS NULL) / COUNT(*) AS null_CAS_ratio,
    COUNTIF(TAS IS NULL) / COUNT(*) AS null_TAS_ratio,
    COUNTIF(latitude IS NULL) / COUNT(*) AS null_latitude_ratio,
    COUNTIF(longitude IS NULL) / COUNT(*) AS null_longitude_ratio,
    FROM `prc-data-challenge-2025.prc2025.flights_train`
    """
)

Null ratio over all trajectory points:
- altitude, latitude, longitude: ~0% -> almost always present (position message).
- groundspeed, vertical_rate, track: 31.6% each, exactly the same -> come together (velocity message). Easy to fill from neighbours (1-2 s apart).
- mach, CAS, TAS: >99.9% null -> unusable as is. Airspeed matters for fuel: will have to be estimated (e.g. from groundspeed + wind).

In [ ]:
# points and airspeed availability per source
q(
    """
    SELECT source, COUNT(*) AS count,
    COUNTIF(mach IS NOT NULL) / COUNT(*) AS mach_ratio,
    COUNTIF(TAS IS NOT NULL) / COUNT(*) AS tas_ratio,
    COUNTIF(CAS IS NOT NULL) / COUNT(*) AS cas_ratio,
    FROM `prc-data-challenge-2025.prc2025.flights_train`
    GROUP BY source
    ORDER BY count DESC
    """
)

source: adsb 123.45M points (99.9%), acars 133k (0.1%).
mach / TAS / CAS come only from acars (adsb: 0%; acars: mach 72%, TAS 38%, CAS 18%).
-> airspeed is available only on rare acars points: not usable as a regular feature.